# Maritime AIS Anomaly Detection LSTM Autoencoder

This notebook covers Layer 2 of the project: data loading, cleaning, feature scaling, sequence building, and the LSTM-autoencoder model used for unsupervised anomaly detection.

Training code is shown for transparency, but all downstream scoring/validation loads the already-trained, saved model (`../models/ais_lstm_autoencoder.keras`) so results stay reproducible across re-runs.

## 1. Load & Inspect Data

In [24]:
import numpy as np 
import pandas as pd
from sklearn.preprocessing import StandardScaler


In [ ]:
featured_data = pd.read_csv("../data/gulf_features.csv")
featured_data.head(10) 

,mmsi,base_datetime,ping_sequence,lat,lon,cog,heading,sog,time_differ,dist
0,83560169,2023-01-19 15:40:12,1,29.54868,-89.77021,253.3,315,3.2,NaN,NaN
1,209130856,2023-01-07 00:12:25,1,29.46187,-89.59770,186.1,448,0.0,NaN,NaN
2,209497000,2023-01-15 17:54:50,1,27.50247,-88.57090,332.0,333,13.0,NaN,NaN
3,209497000,2023-01-15 17:57:20,2,27.51030,-88.57557,332.0,333,12.9,2.500000,0.984976
4,209497000,2023-01-15 17:58:30,3,27.51397,-88.57775,331.0,333,12.9,1.166667,0.461254
5,209497000,2023-01-15 17:59:40,4,27.51782,-88.58002,332.0,333,12.9,1.166667,0.483098
6,209497000,2023-01-15 18:00:40,5,27.52097,-88.58187,331.0,333,12.9,1.000000,0.394927
7,209497000,2023-01-15 18:05:51,6,27.53758,-88.59157,332.0,333,12.9,5.183333,2.079915
8,209497000,2023-01-15 18:11:40,7,27.55633,-88.60222,333.0,333,12.9,5.816667,2.334367
9,209497000,2023-01-15 18:12:51,8,27.56017,-88.60447,333.0,333,12.9,1.183333,0.481161


In [3]:
print(featured_data.shape)
print(featured_data.dtypes)
print(featured_data.isnull().sum())

(5185360, 10)
mmsi               int64
base_datetime     object
ping_sequence      int64
lat              float64
lon              float64
cog              float64
heading            int64
sog              float64
time_differ      float64
dist             float64
dtype: object
mmsi               0
base_datetime      0
ping_sequence      0
lat                0
lon                0
cog                0
heading            0
sog                0
time_differ      300
dist             300
dtype: int64


## 2. Clean Data

Convert `base_datetime` to a real datetime, check per-vessel ping counts, and drop rows with no previous ping to diff against (the first ping of each vessel — 22 of the original 300 sampled vessels had only a single ping and are fully excluded here).

In [4]:
featured_data['base_datetime'] = pd.to_datetime(featured_data['base_datetime'])
print(featured_data.dtypes['base_datetime'])

datetime64[ns]


In [5]:
featured_data.groupby('mmsi').size().sort_values()

mmsi
83560169          1
209130856         1
321687710         1
338196992         1
333950458         1
              ...  
367300080    105386
366997820    105556
366993660    106625
368011470    107120
367305890    107788
Length: 300, dtype: int64

In [6]:
featured_data = featured_data.dropna()

In [7]:
print(featured_data.isnull().sum())

mmsi             0
base_datetime    0
ping_sequence    0
lat              0
lon              0
cog              0
heading          0
sog              0
time_differ      0
dist             0
dtype: int64


In [8]:
print(featured_data.shape)

(5185060, 10)


In [9]:
featured_data.groupby('mmsi').size()

mmsi
209497000    2223
209961000    7337
210296000      36
212287000     569
229214000    6619
             ... 
636022337    1583
636022536    2444
636092848    1074
636093086     730
941217433       5
Length: 278, dtype: int64

## 3. Feature Scaling

`dist` and `time_differ` contain extreme outliers (genuine multi-day AIS silence gaps). A `log1p` transform compresses their scale while preserving relative ordering — so a real anomaly still reads as unusually large, without destabilizing training — before StandardScaler is applied across all 6 features.

In [10]:
featured_data['dist_log'] = np.log1p(featured_data['dist'])
featured_data['time_differ_log'] = np.log1p(featured_data['time_differ'])

scaler = StandardScaler()

In [11]:
raw_data_headings= ['lat','lon','cog','sog','time_differ_log','dist_log']


In [12]:
data_scaled_headings= ['lat_scaled','lon_scaled','cog_scaled','sog_scaled','time_differ_scaled','dist_scaled']


In [13]:
featured_data[data_scaled_headings]=scaler.fit_transform(featured_data[raw_data_headings])

In [14]:
featured_data = featured_data.reset_index(drop=True)

In [15]:
featured_data.index

RangeIndex(start=0, stop=5185060, step=1)

## 4. Build Sequences

Each vessel's pings are chunked into non-overlapping 50-ping sequences (dropping any leftover partial window), tracked alongside their vessel ID and time range so results can be traced back to a specific vessel/window later.

In [16]:
sequence_length = 50
feature_cols = data_scaled_headings 
print(feature_cols)
print(type(feature_cols))


['lat_scaled', 'lon_scaled', 'cog_scaled', 'sog_scaled', 'time_differ_scaled', 'dist_scaled']
<class 'list'>


Sanity check on a single vessel before running the full loop. (This step caught a `MemoryError` at full scale, traced to repeated pandas `.iloc` slicing inside the loop — fixed by converting each group to a numpy array once before slicing.)

In [17]:
test_group = featured_data[featured_data['mmsi'] == 209497000]
test_chunk = test_group[feature_cols].iloc[0:50]
print(test_chunk.shape)

(50, 6)


In [18]:
sequence_length = 50
feature_cols = data_scaled_headings

all_sequences = []
sequence_mmsi = []
sequence_start_time = []
sequence_end_time = []

for mmsi, group in featured_data.groupby('mmsi'):
    group = group.sort_values('base_datetime')
    group_array = np.array(group[feature_cols])
    time_array = group['base_datetime'].values

    n_rows = len(group_array)
    n_full_chunks = n_rows // sequence_length

    for i in range(n_full_chunks):
        start = i * sequence_length
        end = start + sequence_length
        chunk = group_array[start:end]

        all_sequences.append(chunk)
        sequence_mmsi.append(mmsi)
        sequence_start_time.append(time_array[start])
        sequence_end_time.append(time_array[end - 1])

X = np.array(all_sequences)
print(X.shape)

(103566, 50, 6)


## 5. LSTM-Autoencoder Model

Encoder LSTM → bottleneck → `RepeatVector` → Decoder LSTM → `TimeDistributed(Dense)`. Reconstruction error (MSE) on an unseen sequence becomes its anomaly score.

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:


model = Sequential([
    LSTM( 32, activation='relu', input_shape=(50, 6)),   
    RepeatVector(50),                                        
    LSTM(32, activation='relu', return_sequences=True),       
    TimeDistributed(Dense(6))                                 
])

from tensorflow.keras.optimizers import Adam
model.compile(optimizer=Adam(learning_rate=0.0001), loss='mse')
model.summary()

### Training

The cell below is the exact training call used to produce the saved model. It's shown as a fenced code block rather than a live cell, since re-running training produces a different model (different random weight initialization) than the one validated in the Results section. To retrain from scratch, copy this into a new code cell and run it; otherwise, continue to the next section, which loads the already-trained, saved model.

```python
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X, X,
    epochs=200,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop]
)
```

## 6. Load Trained Model & Score Sequences

In [21]:
from tensorflow.keras.models import load_model

model = load_model("../models/ais_lstm_autoencoder.keras")
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 32)             │         4,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_1 (RepeatVector)  │ (None, 50, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 50, 32)         │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 50, 6)          │           198 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,532 (158.33 KB)

 Trainable params: 13,510 (52.77 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 27,022 (105.56 KB)

In [22]:
x_restructured=model.predict(X)

3237/3237 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step


## 7. Results — Anomaly Scores

Reconstruction error per sequence, ranked descending. See the project README for cross-validation against the SQL rule-based flags.

In [25]:

reconstruction_errors = np.mean((X - x_restructured) ** 2, axis=(1, 2))

results = pd.DataFrame({
    'mmsi': sequence_mmsi,
    'start_time': sequence_start_time,
    'end_time': sequence_end_time,
    'anomaly_score': reconstruction_errors
})
results.sort_values('anomaly_score', ascending=False).head(20)

,mmsi,start_time,end_time,anomaly_score
9867,366930000,2023-03-28 07:18:19,2023-03-28 09:48:18,19.313894
6807,366843420,2023-01-08 07:50:06,2023-02-24 00:44:31,14.167660
52064,367501540,2023-01-30 13:48:39,2023-02-28 20:44:26,14.113730
103557,636093086,2023-01-22 11:43:52,2023-03-06 14:56:12,12.165352
54150,367551640,2023-02-08 07:24:34,2023-02-25 13:08:57,11.785872
71082,367764520,2023-02-10 01:19:42,2023-02-10 02:16:02,11.575962
3769,338926105,2023-03-01 16:02:05,2023-03-07 18:00:00,11.551630
3773,338926105,2023-03-07 20:50:24,2023-03-15 13:39:48,11.400285
52066,367501540,2023-03-01 02:22:56,2023-03-08 19:55:48,11.313000
9535,366930000,2023-02-17 17:17:01,2023-02-17 18:42:18,11.038173
